# GRPO Training — Drug Discovery Sim Env (OpenEnv)

End-to-end Colab notebook: install deps, start the env in a background process, run live-rollout GRPO, and produce loss/reward/baseline-vs-trained plots.

Switch to a T4 (or better) runtime before running.

In [ ]:
!pip install -U pip setuptools wheel
!pip install -e .[training,test]
# Optional: Unsloth (much faster on T4 / A100). Skip if it fails — we fall back to plain HF.
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' || true

## 1. Reproducible offline GRPO experiment (smoke test + plots)
Generates loss/reward/baseline-vs-trained plots from a short run.

In [ ]:
!python -m drug_discovery_env.scripts.run_training_experiment \
  --episodes 2 --data-mode hybrid --device auto \
  --model Qwen/Qwen2.5-0.5B-Instruct --max-train-steps 10 \
  --out-dir artifacts/training

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/training/loss_curve.png'))
display(Image('artifacts/training/reward_curve.png'))
display(Image('artifacts/training/baseline_vs_trained.png'))

## 2. Live-rollout GRPO with Qwen2.5-3B-Instruct
Starts the env server in the background, then runs full GRPO training where each rollout calls the live env and uses the env's actual reward.

In [ ]:
import subprocess, time
proc = subprocess.Popen(
    ['uvicorn', 'drug_discovery_env.server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(8)
print('env server started, pid:', proc.pid)

In [ ]:
!python -m drug_discovery_env.scripts.train_grpo_live \
  --base-url http://localhost:8000 \
  --model Qwen/Qwen2.5-3B-Instruct \
  --output-dir outputs/grpo \
  --max-steps 100

In [ ]:
proc.terminate()
print('env server stopped')